In [ ]:
# -*- coding: utf-8 -*-
# stacking_corefe_ts_norec_oofensemb.py

import os
import warnings
import glob  
import re    

warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

# -------------------- Config --------------------
TRAIN_PATH = 'train.csv'
SUB_PATH   = 'sample_submission.csv'
TEST_DIR   = '.' # 이미 파일이 로컬에 있다고 가정

SEEDS = [42, 202, 777]   # base 모델 시드 앙상블
USE_DOW_WEIGHTS = True   # 요일별 가중 메타 허용
N_FOLDS = 3              # OOF 롤링 폴드 수
FOLD_VAL_DAYS = 28       # 폴드당 검증 길이(일)
META_ENSEMBLE_TOPK = 2   # 테스트에서 상위 k 개 메타 평균 (1이면 단일 best만)

# -------------------- Optional deps --------------------
try:
    from scipy.optimize import nnls as scipy_nnls, minimize
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False

# -------------------- Holidays --------------------
HOLIDAYS = {
    '2023-01-21','2023-01-22','2023-01-23','2023-01-24','2023-03-01','2023-05-05','2023-05-27','2023-05-29',
    '2023-06-06','2023-08-15','2023-09-28','2023-09-29','2023-09-30','2023-10-01','2023-10-02','2023-10-03',
    '2023-10-09','2023-12-25','2024-01-01','2024-02-09','2024-02-10','2024-02-11','2024-02-12','2024-03-01',
    '2024-04-10','2024-05-05','2024-05-06','2024-05-15','2024-06-06','2024-08-15','2024-09-16','2024-09-17',
    '2024-09-18','2024-10-03','2024-10-09','2024-12-25','2025-01-01','2025-01-28','2025-01-29','2025-01-30',
    '2025-03-01','2025-05-05','2025-05-06'
}

# -------------------- Metrics --------------------
def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    if len(y_true) == 0:
        return 0.0
    denom = (np.abs(y_true) + np.abs(y_pred)).clip(min=1e-8)
    return np.mean(2.0 * np.abs(y_pred - y_true) / denom)

store_weights = {'담하': 1.3, '미라시아': 1.3}
default_weight = 1.0

def weighted_smape(y_true, y_pred, weights):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = y_true != 0
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    weights = weights[mask]
    if len(y_true) == 0:
        return 0.0
    denom = (np.abs(y_true) + np.abs(y_pred)).clip(min=1e-8)
    smape_val = 2.0 * np.abs(y_pred - y_true) / denom
    return np.average(smape_val, weights=weights)

def get_store_weights(store_series):
    return np.array([store_weights.get(s, default_weight) for s in store_series], dtype=float)

def weighted_smape_eval(X_val, store_col='영업장명'):
    stores_val = X_val[store_col].values
    weights_val = np.array([store_weights.get(s, default_weight) for s in stores_val], dtype=float)
    def eval_metric(y_true, y_pred):
        y_true_np = np.asarray(y_true)
        score = weighted_smape(y_true, y_pred, weights_val) if y_true_np.shape[0] == weights_val.shape[0] else smape(y_true, y_pred)
        return 'wSMAPE', score, False
    return eval_metric

# -------------------- Calendar Features --------------------
def add_features(df):
    df_copy = df.copy()
    df_copy['영업일자'] = pd.to_datetime(df_copy['영업일자'])

    # split "영업장명_메뉴명"
    store_names, menu_names = zip(*[
        (str(sm).split('_', 1)[0], str(sm).split('_', 1)[1] if '_' in str(sm) else '')
        for sm in df_copy['영업장명_메뉴명']
    ])
    df_copy['영업장명'] = store_names
    df_copy['메뉴명']   = menu_names

    # date parts
    df_copy['year']        = df_copy['영업일자'].dt.year
    df_copy['month']       = df_copy['영업일자'].dt.month
    df_copy['day']         = df_copy['영업일자'].dt.day
    df_copy['day_of_week'] = df_copy['영업일자'].dt.dayofweek
    df_copy['week_of_year']= df_copy['영업일자'].dt.isocalendar().week.astype(int)

    # holiday/weekend/sandwich
    holidays = set(HOLIDAYS)
    unique_dates = pd.DataFrame({'영업일자': df_copy['영업일자'].unique()}).sort_values('영업일자')
    unique_dates['요일'] = unique_dates['영업일자'].dt.weekday
    unique_dates['holiday_info'] = 0
    is_public_holiday = unique_dates['영업일자'].dt.date.astype(str).isin(holidays)
    unique_dates.loc[is_public_holiday, 'holiday_info'] = 3
    is_weekend = (unique_dates['요일'] >= 5) & (unique_dates['holiday_info'] == 0)
    unique_dates.loc[is_weekend, 'holiday_info'] = 2
    prev_day_holiday = unique_dates['holiday_info'].shift(1).fillna(0)
    next_day_holiday = unique_dates['holiday_info'].shift(-1).fillna(0)
    is_sandwich_day = (unique_dates['holiday_info'] == 0) & (prev_day_holiday >= 2) & (next_day_holiday >= 2)
    unique_dates.loc[is_sandwich_day, 'holiday_info'] = 1
    df_copy = pd.merge(df_copy, unique_dates[['영업일자','holiday_info']], on='영업일자', how='left')

    # distance to holiday
    holiday_dt = pd.to_datetime(sorted(HOLIDAYS)).to_numpy(dtype='datetime64[D]')
    dts = df_copy['영업일자'].to_numpy(dtype='datetime64[D]')
    if holiday_dt.size == 0:
        df_copy['is_before_holiday'] = 0
        df_copy['is_after_holiday']  = 0
        df_copy['days_to_holiday']   = 8
    else:
        idx_next = np.searchsorted(holiday_dt, dts, side='left')
        idx_next_clip = np.minimum(idx_next, holiday_dt.size - 1)
        delta_next = (holiday_dt[idx_next_clip] - dts).astype('timedelta64[D]').astype(int)
        days_to_next = np.where(idx_next < holiday_dt.size, delta_next, 9999)

        idx_prev = np.maximum(idx_next - 1, 0)
        prev_is_valid = holiday_dt[idx_prev] <= dts
        delta_prev = (dts - holiday_dt[idx_prev]).astype('timedelta64[D]').astype(int)
        days_from_prev = np.where(prev_is_valid, delta_prev, 9999)

        df_copy['is_before_holiday'] = (days_to_next == 1).astype(int)
        df_copy['is_after_holiday']  = (days_from_prev == 1).astype(int)

        dist_raw = np.minimum(days_to_next, days_from_prev)
        df_copy['days_to_holiday'] = np.where(dist_raw == 9999, 8, np.minimum(dist_raw, 7))

    return df_copy

def set_categorical_features(df, categorical_cols):
    df = df.copy()
    for col in categorical_cols:
        df[col] = df[col].astype('category')
    return df

# -------------------- TS Features (strictly past) --------------------
def add_ts_features(df):
    """
    입력 df: ['영업일자','영업장명','메뉴명','매출수량', ...]
    반환: lag/rolling/ewm/trend/last_level/dow_count 등 추가 (shift 사용으로 미래 누설 방지)
    """
    df = df.sort_values(['영업장명','메뉴명','영업일자']).copy()
    key = ['영업장명','메뉴명']

    # lags
    for l in [1, 7, 14]:
        df[f'lag{l}'] = df.groupby(key)['매출수량'].shift(l)

    # rolling means
    df['roll7']  = df.groupby(key)['매출수량'].shift(1).rolling(7).mean()
    df['roll14'] = df.groupby(key)['매출수량'].shift(1).rolling(14).mean()
    df['roll28'] = df.groupby(key)['매출수량'].shift(1).rolling(28).mean()

    # DOW-wise rolling mean & count
    df['dow_mean_28'] = (
        df.groupby(key + ['day_of_week'])['매출수량'].shift(1).rolling(28).mean()
    )
    df['dow_count_28'] = (
        df.groupby(key + ['day_of_week'])['매출수량'].apply(lambda s: s.shift(1).rolling(28).count())
    ).values

    # EWM (exponential)
    df['ewm7']  = df.groupby(key)['매출수량'].shift(1).ewm(alpha=2/(7+1), adjust=False).mean()
    df['ewm14'] = df.groupby(key)['매출수량'].shift(1).ewm(alpha=2/(14+1), adjust=False).mean()

    # trend / last level / ratio
    eps = 1e-6
    df['trend_7_14']   = (df['roll7'] + eps) / (df['roll14'] + eps)
    df['trend_7_28']   = (df['roll7'] + eps) / (df['roll28'] + eps)
    df['last_level']   = df.groupby(key)['매출수량'].shift(1).fillna(0.0)
    df['last_vs_roll7'] = (df['last_level'] + eps) / (df['roll7'] + eps)

    # fillna
    for c in ['lag1','lag7','lag14','roll7','roll14','roll28','dow_mean_28','dow_count_28',
              'ewm7','ewm14','trend_7_14','trend_7_28','last_level','last_vs_roll7']:
        df[c] = df[c].fillna(0.0)

    return df

# -------------------- Submission --------------------
def convert_to_submission_format(pred_df: pd.DataFrame, sample_submission: pd.DataFrame):
    pred_dict = dict(zip(zip(pred_df['영업일자'], pred_df['영업장명_메뉴명']), pred_df['매출수량']))
    final_df = sample_submission.copy()
    for row_idx in final_df.index:
        date = final_df.loc[row_idx, '영업일자']
        for col in final_df.columns[1:]:
            final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)
    return final_df

# -------------------- DOW 가중 학습 --------------------
def learn_dow_weights(A, B, y_true, dows, weights, n_grid=2001, base_w=0.5, step=4):
    ws = np.linspace(0.0, 1.0, n_grid)
    w_map = {}
    for d in range(7):
        m = (dows == d)
        if m.sum() == 0:
            w_map[d] = base_w
            continue
        best = (base_w, 1e18)
        for w in ws[::step]:
            s = weighted_smape(y_true[m], w*A[m] + (1-w)*B[m], weights[m])
            if s < best[1]:
                best = (w, s)
        w_map[d] = float(best[0])
    return w_map

# -------------------- Base models --------------------
def predict_blend_A(models_tweedie, models_poisson, X, w_poisson=0.3, w_tweedie=0.7):
    preds_p = [np.clip(m.predict(X, num_iteration=m.best_iteration_), 0, None) for m in models_poisson]
    preds_t = [np.clip(m.predict(X, num_iteration=m.best_iteration_), 0, None) for m in models_tweedie]
    return w_poisson * np.mean(preds_p, axis=0) + w_tweedie * np.mean(preds_t, axis=0)

def predict_B(models_alt, X):
    preds = [np.clip(m.predict(X, num_iteration=m.best_iteration_), 0, None) for m in models_alt]
    return np.mean(preds, axis=0)

# -------------------- No-recursion future feature builder --------------------
def build_future_features_no_recursion(curr_test, learned_categories, FEATURES):
    """
    재귀 없이 7일치 피처를 한 번에 구성.
    과거에서 계산된 시계열 피처의 '마지막 값'을 스냅샷으로 가져와 7일 모두 동일 적용.
    """
    hist = curr_test.copy()
    if '매출수량' not in hist.columns:
        hist['매출수량'] = 0.0
    hist = add_features(hist)
    hist = add_ts_features(hist)

    hist_sorted = hist.sort_values('영업일자')
    key_cols = ['영업장명','메뉴명']

    # 그룹 마지막 스냅샷(roll/lag/ewm/trend/last 등)
    snap_cols = ['lag1','lag7','lag14','roll7','roll14','roll28',
                 'ewm7','ewm14','trend_7_14','trend_7_28','last_level','last_vs_roll7']
    last_snap = (
        hist_sorted.groupby(key_cols, as_index=False)
        .tail(1)[key_cols + snap_cols]
        .set_index(key_cols)
    )

    # 그룹×요일별 dow_mean_28, dow_count_28 마지막 관측치
    hist_sorted['__dow'] = hist_sorted['day_of_week']
    last_dow = (
        hist_sorted.groupby(key_cols + ['__dow'], as_index=False)
        .tail(1)[key_cols + ['__dow', 'dow_mean_28','dow_count_28']]
        .set_index(key_cols + ['__dow'])
    )

    # 7일 대상 프레임
    last_day = pd.to_datetime(curr_test['영업일자']).max()
    future_dates = pd.date_range(start=last_day + pd.Timedelta(days=1), periods=7, freq='D')
    store_menus = curr_test['영업장명_메뉴명'].unique()
    fut_idx = pd.MultiIndex.from_product([future_dates, store_menus],
                                         names=['영업일자','영업장명_메뉴명'])
    fut = pd.DataFrame(index=fut_idx).reset_index()
    fut = add_features(fut)

    # 카테고리 고정(1차)
    fut['영업장명'] = pd.Categorical(fut['영업장명'], categories=learned_categories['영업장명'])
    fut['메뉴명']   = pd.Categorical(fut['메뉴명'],   categories=learned_categories['메뉴명'])

    # 스냅샷 병합 (group-level)
    fut = fut.merge(last_snap.reset_index(), on=['영업장명','메뉴명'], how='left')

    # 요일 스냅샷 병합 (group×dow)
    fut['__dow'] = fut['day_of_week']
    fut = fut.merge(
        last_dow.reset_index().rename(columns={'__dow':'__dow_key'}),
        left_on=['영업장명','메뉴명','__dow'],
        right_on=['영업장명','메뉴명','__dow_key'],
        how='left'
    ).drop(columns=['__dow_key'])

    # merge 이후 카테고리 재고정(2차)
    fut['영업장명'] = pd.Categorical(fut['영업장명'], categories=learned_categories['영업장명'])
    fut['메뉴명']   = pd.Categorical(fut['메뉴명'],   categories=learned_categories['메뉴명'])

    # 결측 보정
    fill_cols = snap_cols + ['dow_mean_28','dow_count_28']
    for c in fill_cols:
        if c not in fut.columns:
            fut[c] = 0.0
        else:
            fut[c] = fut[c].fillna(0.0)

    X_inf = fut[FEATURES].copy()
    k_days = (fut['영업일자'] - (last_day + pd.Timedelta(days=0))).dt.days.values
    return X_inf, k_days, fut['영업장명_메뉴명'].values

# -------------------- TS folds --------------------
def make_ts_folds(df_dates, n_folds=3, val_len=28):
    last_day = pd.to_datetime(df_dates.max())
    folds = []
    for k in range(n_folds, 0, -1):
        val_end   = last_day - pd.Timedelta(days=(k-1)*val_len)
        val_start = val_end - pd.Timedelta(days=val_len-1)
        folds.append((val_start.normalize(), val_end.normalize()))
    return folds

# -------------------- Meta application helper --------------------
def apply_meta(meta_name, meta_models, X2, ctx=None):
    A2, B2 = X2[:,0], X2[:,1]
    if meta_name == 'elasticnet':   return meta_models['elasticnet'].predict(X2)
    if meta_name == 'ridge':        return meta_models['ridge'].predict(X2)
    if meta_name == 'lgbm':         return meta_models['lgbm'].predict(X2)
    if meta_name == 'bayesridge':   return meta_models['bayesridge'].predict(X2)
    if meta_name == 'tweedie':      return meta_models['tweedie'].predict(X2)
    if meta_name == 'huber':        return meta_models['huber'].predict(X2)
    if meta_name == 'nnls':
        w = meta_models['nnls_weights']; return X2 @ w
    if meta_name == 'smape_opt':
        w = meta_models['smape_opt_weight']; return w*A2 + (1-w)*B2
    if meta_name == 'iso_on_smapeopt':
        w, iso = meta_models['iso_on_smapeopt']; z = w*A2 + (1-w)*B2; return iso.predict(z)
    if meta_name == 'dow_smape_opt':
        wmap = meta_models['dow_smape_weights']
        dows = ctx['day_of_week'].values if ctx is not None else None
        ww = np.array([wmap.get(int(d), meta_models.get('smape_opt_weight', 0.5)) for d in dows], float)
        return ww*A2 + (1-ww)*B2
    if meta_name == 'logistic_gate':
        a, b, c = meta_models['logistic_gate']
        wA = 1.0/(1.0 + np.exp(-(a*A2 + b*B2 + c)))
        return wA*A2 + (1-wA)*B2
    return 0.5*A2 + 0.5*B2

# -------------------- Main --------------------
def main():
    # 1) Load & FE
    raw = pd.read_csv(TRAIN_PATH)
    df  = add_features(raw)
    df  = set_categorical_features(df, ['영업장명','메뉴명','영업장명_메뉴명'])
    df['매출수량'] = df['매출수량'].clip(lower=0)

    # TS features (past only)
    df = add_ts_features(df)

    FEATURES = [
        'month','day','day_of_week','메뉴명','영업장명','week_of_year',
        'holiday_info','is_before_holiday','is_after_holiday','days_to_holiday',
        'lag1','lag7','lag14','roll7','roll14','roll28','dow_mean_28','dow_count_28',
        'ewm7','ewm14','trend_7_14','trend_7_28','last_level','last_vs_roll7'
    ]
    TARGET = '매출수량'

    # 2) Single holdout for early-stopping only (last 56d)
    last_date = df['영업일자'].max()
    cutoff = pd.to_datetime(last_date) - pd.Timedelta(days=56)
    train_data = df[df['영업일자'] < cutoff].copy()
    valid_data = df[df['영업일자'] >= cutoff].copy()

    X_train = train_data[FEATURES]
    y_train = train_data[TARGET].values
    X_valid = valid_data[FEATURES]
    y_valid = valid_data[TARGET].values
    eval_metric = weighted_smape_eval(X_valid, store_col='영업장명')

    # 3) Train base models (once)
    base_params = {
        'n_estimators': 6000,
        'subsample_freq': 2,
        'learning_rate': 0.03,
        'num_leaves': 128,
        'max_depth': 10,
        'subsample': 0.9,
        'colsample_bytree': 0.9,
        'reg_alpha': 1e-2,
        'reg_lambda': 1e-2,
        'min_data_in_leaf': 64,        # generalization
        'min_child_samples': 64,
        'extra_trees': True,
        'verbosity': -1
    }
    models_tweedie, models_poisson = [], []
    for seed in SEEDS:
        m_t = LGBMRegressor(**{**base_params, 'objective':'tweedie', 'tweedie_variance_power':1.5, 'random_state':seed})
        m_t.fit(X_train, y_train, eval_set=[(X_train,y_train),(X_valid,y_valid)],
                eval_metric=eval_metric, callbacks=[early_stopping(300), log_evaluation(100)])
        models_tweedie.append(m_t)

        m_p = LGBMRegressor(**{**base_params, 'objective':'poisson', 'random_state':seed})
        m_p.fit(X_train, y_train, eval_set=[(X_train,y_train),(X_valid,y_valid)],
                eval_metric=eval_metric, callbacks=[early_stopping(300), log_evaluation(100)])
        models_poisson.append(m_p)

    alt_params = {
        'objective': 'regression',
        'n_estimators': 6000,
        'learning_rate': 0.02,
        'num_leaves': 64,
        'max_depth': 8,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'reg_alpha': 1e-3,
        'reg_lambda': 1e-2,
        'min_data_in_leaf': 64,
        'min_child_samples': 64,
        'extra_trees': True,
        'verbosity': -1
    }
    models_alt = []
    for seed in SEEDS:
        m = LGBMRegressor(**{**alt_params, 'random_state':seed})
        m.fit(X_train, y_train, eval_set=[(X_train,y_train),(X_valid,y_valid)],
              eval_metric=eval_metric, callbacks=[early_stopping(200), log_evaluation(100)])
        models_alt.append(m)

    # 4) Build OOF for meta via rolling folds
    folds = make_ts_folds(df['영업일자'], n_folds=N_FOLDS, val_len=FOLD_VAL_DAYS)

    meta_oof = []
    meta_y   = []
    meta_w   = []
    meta_ctx = []
    meta_keys= []

    for (vs, ve) in folds:
        trn = df[(df['영업일자'] < vs)].copy()
        val = df[(df['영업일자'] >= vs) & (df['영업일자'] <= ve)].copy()
        if len(trn) == 0 or len(val) == 0:
            continue

        X_val = val[FEATURES]; y_val = val[TARGET].values
        pA = predict_blend_A(models_tweedie, models_poisson, X_val)
        pB = predict_B(models_alt, X_val)

        meta_oof.append(np.vstack([pA, pB]).T)
        meta_y.append(y_val)
        meta_w.append(get_store_weights(val['영업장명']))
        meta_ctx.append(val[['day_of_week']].reset_index(drop=True))
        meta_keys.append(val[['영업장명_메뉴명']].reset_index(drop=True))

    X_meta_valid = np.vstack(meta_oof)
    y_meta_valid = np.concatenate(meta_y)
    w_val_all    = np.concatenate(meta_w)
    ctx_valid    = pd.concat(meta_ctx, axis=0, ignore_index=True)
    keys_valid   = pd.concat(meta_keys, axis=0, ignore_index=True)['영업장명_메뉴명'].values

    # 5) Train meta models & compare
    meta_models, meta_scores = {}, {}

    try:
        from sklearn.pipeline import Pipeline
        from sklearn.preprocessing import StandardScaler
        from sklearn.linear_model import ElasticNetCV, RidgeCV, BayesianRidge, HuberRegressor, TweedieRegressor

        pipe_en = Pipeline([("scaler", StandardScaler()),
                             ("enet",   ElasticNetCV(l1_ratio=[0.1,0.3,0.5,0.7,0.9], cv=5, n_jobs=-1, max_iter=30000))])
        pipe_en.fit(X_meta_valid, y_meta_valid)
        meta_models['elasticnet'] = pipe_en
        meta_scores['elasticnet']  = weighted_smape(y_meta_valid, pipe_en.predict(X_meta_valid), w_val_all)

        pipe_rg = Pipeline([("scaler", StandardScaler()),
                             ("ridge",  RidgeCV(alphas=np.logspace(-3,3,25), cv=5))])
        pipe_rg.fit(X_meta_valid, y_meta_valid)
        meta_models['ridge'] = pipe_rg
        meta_scores['ridge']  = weighted_smape(y_meta_valid, pipe_rg.predict(X_meta_valid), w_val_all)

        meta_lgbm = LGBMRegressor(n_estimators=1000, max_depth=3, learning_rate=0.05,
                                  subsample=0.9, colsample_bytree=1.0, random_state=7)
        meta_lgbm.fit(X_meta_valid, y_meta_valid)
        meta_models['lgbm'] = meta_lgbm
        meta_scores['lgbm']  = weighted_smape(y_meta_valid, meta_lgbm.predict(X_meta_valid), w_val_all)

        br = BayesianRidge()
        br.fit(X_meta_valid, y_meta_valid)
        meta_models['bayesridge'] = br
        meta_scores['bayesridge']  = weighted_smape(y_meta_valid, br.predict(X_meta_valid), w_val_all)

        best_tr, best_tr_s = None, 1e18
        for pwr in [1.1, 1.3, 1.5, 1.7]:
            tr = TweedieRegressor(power=pwr, alpha=0.0, max_iter=5000)
            tr.fit(X_meta_valid, y_meta_valid)
            s = weighted_smape(y_meta_valid, tr.predict(X_meta_valid), w_val_all)
            if s < best_tr_s:
                best_tr_s, best_tr = s, tr
        if best_tr is not None:
            meta_models['tweedie'] = best_tr
            meta_scores['tweedie']  = best_tr_s

        hr = HuberRegressor(epsilon=1.35, max_iter=1000)
        hr.fit(X_meta_valid, y_meta_valid)
        meta_models['huber'] = hr
        meta_scores['huber']  = weighted_smape(y_meta_valid, hr.predict(X_meta_valid), w_val_all)

        APPLY_ISO_LATER = True
    except Exception as e:
        print("Some sklearn meta models failed:", e)
        APPLY_ISO_LATER = False

    # c) smape_opt (scalar)
    ws = np.linspace(0, 1, 2001)
    A, B = X_meta_valid[:,0], X_meta_valid[:,1]
    best_w, best_s = 0.5, 1e18
    for w in ws:
        s = weighted_smape(y_meta_valid, w*A + (1-w)*B, w_val_all)
        if s < best_s:
            best_s, best_w = s, w
    meta_models['smape_opt_weight'] = best_w
    meta_scores['smape_opt'] = best_s

    # d) dow_smape_opt (요일별) OOF 기반
    if USE_DOW_WEIGHTS:
        dows = ctx_valid['day_of_week'].values.astype(int)
        w_dow = learn_dow_weights(A, B, y_meta_valid, dows, w_val_all, n_grid=2001, base_w=best_w, step=4)
        meta_models['dow_smape_weights'] = w_dow
        w_rows = np.array([w_dow.get(int(dd), best_w) for dd in dows], dtype=float)
        blend_dow = np.clip(w_rows*A + (1.0 - w_rows)*B, 0, None)
        meta_scores['dow_smape_opt'] = weighted_smape(y_meta_valid, blend_dow, w_val_all)

    # e) logistic_gate (OOF 기반)
    if HAS_SCIPY:
        try:
            def gate_loss(theta):
                a, b, c = theta
                wA = 1.0 / (1.0 + np.exp(-(a*A + b*B + c)))
                blend = wA*A + (1.0 - wA)*B
                return weighted_smape(y_meta_valid, blend, w_val_all)
            res = minimize(gate_loss, x0=np.zeros(3), method='Nelder-Mead',
                           options={'maxiter':3000, 'xatol':1e-6, 'fatol':1e-6})
            if res.success:
                meta_models['logistic_gate'] = res.x
                meta_scores['logistic_gate'] = gate_loss(res.x)
        except Exception as e:
            print("logistic_gate failed:", e)

    # f) isotonic on smape_opt (1D 보정)
    if APPLY_ISO_LATER and 'smape_opt_weight' in meta_models:
        try:
            from sklearn.isotonic import IsotonicRegression
            w = meta_models['smape_opt_weight']
            z_valid = w*A + (1-w)*B
            iso = IsotonicRegression(out_of_bounds='clip')
            iso.fit(z_valid, y_meta_valid)
            meta_models['iso_on_smapeopt'] = (w, iso)
            meta_scores['iso_on_smapeopt'] = weighted_smape(y_meta_valid, iso.predict(z_valid), w_val_all)
        except Exception as e:
            print("isotonic failed:", e)

    # 6) Pick best meta
    print("\nMeta-model OOF wSMAPE:")
    for k, v in sorted(meta_scores.items(), key=lambda x: x[1]):
        print(f"  {k:15s}: {v:.6f}")
    best_meta_name = min(meta_scores, key=meta_scores.get)
    print(f"\nSelected meta-model: {best_meta_name}")

    # 7) Group scale correction (from OOF) — optional but effective
    yhat_oof = np.clip(apply_meta(best_meta_name, meta_models, X_meta_valid, ctx_valid), 0, None)
    grp_sum_y = {}
    grp_sum_p = {}
    for k, yt, yp in zip(keys_valid, y_meta_valid, yhat_oof):
        grp_sum_y[k] = grp_sum_y.get(k, 0.0) + float(yt)
        grp_sum_p[k] = grp_sum_p.get(k, 0.0) + float(yp)
    scale_table = {k: np.clip((grp_sum_y[k] / max(grp_sum_p[k], 1e-6)), 0.7, 1.4) for k in grp_sum_y}
    meta_models['group_scale'] = scale_table

    # 8) Test prediction (no recursion)
    sample_submission = pd.read_csv(SUB_PATH)
    pred_frames = []

    learned_categories = {c: X_train[c].cat.categories for c in ['영업장명', '메뉴명']}

    test_files = sorted(glob.glob(os.path.join(TEST_DIR, 'TEST_*.csv')))

    if not test_files:
        raise RuntimeError(f"'{TEST_DIR}' 디렉토리에서 'TEST_*.csv' 패턴의 테스트 파일을 찾을 수 없습니다.")

    for test_file in test_files:
        print(f"Processing {test_file}...")

        try:
            basename = os.path.basename(test_file)
            # 'TEST_08.csv' 에서 숫자 '8'을 찾아 정수형으로 변환
            i = int(re.search(r'(\d+)', basename).group(1))
        except (AttributeError, ValueError):
            print(f"Warning: 파일명 '{basename}'에서 테스트 인덱스를 추출할 수 없습니다. 이 파일을 건너뜁니다.")
            continue

        curr_test = pd.read_csv(test_file)

        X_inf, k_days, sm_keys = build_future_features_no_recursion(
            curr_test, learned_categories, FEATURES
        )
        # safety: categorical re-fix
        X_inf['영업장명'] = pd.Categorical(X_inf['영업장명'], categories=learned_categories['영업장명'])
        X_inf['메뉴명']   = pd.Categorical(X_inf['메뉴명'],   categories=learned_categories['메뉴명'])

        # Base
        predA_test = predict_blend_A(models_tweedie, models_poisson, X_inf)
        predB_test = predict_B(models_alt, X_inf)
        Xm = np.vstack([predA_test, predB_test]).T

        # Meta: top-K ensemble or best only
        if META_ENSEMBLE_TOPK >= 2:
            topk = sorted(meta_scores.items(), key=lambda x: x[1])[:META_ENSEMBLE_TOPK]
            preds = []
            for name, _ in topk:
                preds.append(apply_meta(name, meta_models, Xm, ctx=X_inf[['day_of_week']]))
            yhat = np.mean(preds, axis=0)
        else:
            yhat = apply_meta(best_meta_name, meta_models, Xm, ctx=X_inf[['day_of_week']])

        # Group scale correction
        if 'group_scale' in meta_models:
            scales = np.array([meta_models['group_scale'].get(k, 1.0) for k in sm_keys], dtype=float)
            yhat = yhat * scales

        yhat = np.clip(np.asarray(yhat).ravel(), 0.0, None)
        yhat = np.rint(yhat).astype(int)

        # 추출한 테스트 인덱스(i)를 사용하여 제출 파일의 날짜 라벨을 생성
        label_rows = np.array([f"TEST_{i:02d}+{k}일" for k in k_days], dtype=object)
        out_df = pd.DataFrame({
            '영업일자': label_rows,
            '영업장명_메뉴명': sm_keys,
            '매출수량': yhat
        })
        pred_frames.append(out_df[['영업일자','영업장명_메뉴명','매출수량']])

    if not pred_frames:
        # 에러 메시지를 좀 더 일반적인 내용으로 변경
        raise RuntimeError("테스트 파일을 처리했으나 생성된 예측이 없습니다.")

    pred_df = pd.concat(pred_frames, axis=0, ignore_index=True)
    submission = convert_to_submission_format(pred_df, sample_submission)
    submission.to_csv('stacking_corefe_ts_norec_oofensemb_submission.csv', index=False, encoding='utf-8-sig')
    print("\nStacking submission 생성 완료: stacking_corefe_ts_norec_oofensemb_submission.csv")

if __name__ == "__main__":
    main()

/tmp/ipython-input-238741085.py:162: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[f'lag{l}'] = df.groupby(key)['매출수량'].shift(l)
/tmp/ipython-input-238741085.py:162: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[f'lag{l}'] = df.groupby(key)['매출수량'].shift(l)
/tmp/ipython-input-238741085.py:162: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[f'lag{l}'] = df.groupby(key)['매출수량'].shift(l)
/tmp/ipython-input-238741085

Training until validation scores don't improve for 300 rounds
[100]	training's tweedie: 7.97268	training's wSMAPE: 0.532302	valid_1's tweedie: 7.86075	valid_1's wSMAPE: 0.587915
[200]	training's tweedie: 7.69468	training's wSMAPE: 0.536844	valid_1's tweedie: 7.69163	valid_1's wSMAPE: 0.609566
[300]	training's tweedie: 7.5662	training's wSMAPE: 0.522229	valid_1's tweedie: 7.64195	valid_1's wSMAPE: 0.609131
Early stopping, best iteration is:
[63]	training's tweedie: 8.34732	training's wSMAPE: 0.524061	valid_1's tweedie: 8.09464	valid_1's wSMAPE: 0.566028
Training until validation scores don't improve for 300 rounds
[100]	training's poisson: -32.9567	training's wSMAPE: 0.599403	valid_1's poisson: -17.6514	valid_1's wSMAPE: 0.663747
[200]	training's poisson: -34.4575	training's wSMAPE: 0.482048	valid_1's poisson: -18.7589	valid_1's wSMAPE: 0.543912
[300]	training's poisson: -34.9291	training's wSMAPE: 0.511377	valid_1's poisson: -19.0397	valid_1's wSMAPE: 0.580333
[400]	training's poisson:

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/tmp/ipython-input-238741085.py:530: RuntimeWarning: overflow encountered in exp
  wA = 1.0 / (1.0 + np.exp(-(a*A + b*B + c)))



Meta-model OOF wSMAPE:
  logistic_gate  : 0.541124
  dow_smape_opt  : 0.543532
  smape_opt      : 0.544606
  lgbm           : 0.616349
  iso_on_smapeopt: 0.631905
  tweedie        : 0.647570
  huber          : 0.814172
  elasticnet     : 0.933669
  ridge          : 0.949040
  bayesridge     : 0.975840

Selected meta-model: logistic_gate
Processing ./TEST_00.csv...


/tmp/ipython-input-238741085.py:337: RuntimeWarning: overflow encountered in exp
  wA = 1.0/(1.0 + np.exp(-(a*A2 + b*B2 + c)))
/tmp/ipython-input-238741085.py:337: RuntimeWarning: overflow encountered in exp
  wA = 1.0/(1.0 + np.exp(-(a*A2 + b*B2 + c)))


...

Processing ./TEST_08.csv...


/tmp/ipython-input-238741085.py:337: RuntimeWarning: overflow encountered in exp
  wA = 1.0/(1.0 + np.exp(-(a*A2 + b*B2 + c)))


Processing ./TEST_09.csv...


/tmp/ipython-input-238741085.py:337: RuntimeWarning: overflow encountered in exp
  wA = 1.0/(1.0 + np.exp(-(a*A2 + b*B2 + c)))



Stacking submission 생성 완료: stacking_corefe_ts_norec_oofensemb_submission.csv
